# 实验结果分析

本Notebook用于分析和可视化Coreset选择方法的实验结果。

## 主要功能
1. 加载实验结果
2. 数据摘要实验分析
3. 持续学习实验分析
4. 生成比较表格
5. 创建可视化图表

In [ ]:
# 导入必要的库
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional

# 添加项目路径
def get_project_path():
    """自动检测项目路径"""
    if 'google.colab' in sys.modules:
        # Colab环境
        project_path = Path('/content/coreset_benchmark')
    else:
        # 本地环境 - notebook在notebooks目录下
        project_path = Path().absolute().parent
    
    # 添加到sys.path
    if str(project_path) not in sys.path:
        sys.path.insert(0, str(project_path))
    
    return project_path

project_root = get_project_path()

# 设置绘图参数
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# 设置seaborn样式
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.2)

print("库导入成功")
print(f"项目根目录: {project_root}")

## 1. 加载实验结果

In [ ]:
def load_results(log_dir: str, pattern: str = "*.json") -> List[Dict]:
    """
    从日志目录加载实验结果
    
    参数:
        log_dir: 日志目录路径（可以是相对路径或绝对路径）
        pattern: 文件匹配模式
    
    返回:
        结果字典列表
    """
    # 如果是相对路径，则相对于项目根目录
    log_path = Path(log_dir)
    if not log_path.is_absolute():
        log_path = project_root / log_path
    
    if not log_path.exists():
        print(f"警告: 日志目录不存在: {log_dir}")
        print(f"完整路径: {log_path}")
        return []
    
    results = []
    for json_file in log_path.glob(pattern):
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                result = json.load(f)
                result['_source_file'] = str(json_file)
                results.append(result)
        except Exception as e:
            print(f"警告: 无法加载文件 {json_file}: {e}")
    
    return results

# 设置结果目录（相对于项目根目录）
results_dir = project_root / 'results'
data_summ_dir = results_dir / 'data_summarization'
continual_dir = results_dir / 'continual_learning'

print(f"项目根目录: {project_root}")
print(f"结果目录: {results_dir}")
print(f"数据摘要结果目录: {data_summ_dir}")
print(f"持续学习结果目录: {continual_dir}")

In [ ]:
# 加载数据摘要实验结果
data_summ_results = load_results(str(data_summ_dir))
print(f"加载了 {len(data_summ_results)} 个数据摘要实验结果")

# 加载持续学习实验结果
continual_results = load_results(str(continual_dir))
print(f"加载了 {len(continual_results)} 个持续学习实验结果")

## 2. 数据摘要实验分析

In [ ]:
# 将结果转换为DataFrame
if data_summ_results:
    df_data_summ = pd.DataFrame(data_summ_results)
    print("数据摘要实验结果列:")
    print(df_data_summ.columns.tolist())
    print(f"\n结果数量: {len(df_data_summ)}")
    print(f"\n数据集: {df_data_summ['dataset'].unique().tolist()}")
    print(f"方法: {df_data_summ['method'].unique().tolist()}")
else:
    print("没有数据摘要实验结果")
    df_data_summ = pd.DataFrame()

In [ ]:
# 查看基本统计信息
if not df_data_summ.empty:
    print("\n基本统计信息:")
    print(df_data_summ.describe())
    
    # 按数据集和方法分组统计
    print("\n按数据集和方法分组:")
    summary = df_data_summ.groupby(['dataset', 'method']).agg({
        'test_acc_coreset': ['mean', 'std'],
        'performance_drop': ['mean', 'std'],
        'selection_time': ['mean', 'std']
    })
    print(summary)

In [ ]:
# 绘制数据摘要结果比较图
if not df_data_summ.empty:
    datasets = df_data_summ['dataset'].unique()
    
    for dataset in datasets:
        ds_data = df_data_summ[df_data_summ['dataset'] == dataset]
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f'{dataset} 数据集 - Coreset选择方法比较', fontsize=16, fontweight='bold')
        
        methods = ds_data['method'].unique()
        
        # 1. 测试准确率比较
        ax1 = axes[0, 0]
        for method in methods:
            method_data = ds_data[ds_data['method'] == method]
            if len(method_data) > 0:
                ratios = method_data['selection_ratio'] * 100
                accs = method_data['test_acc_coreset']
                ax1.plot(ratios, accs, marker='o', label=method.upper(), linewidth=2)
        
        ax1.set_xlabel('Coreset大小 (%)')
        ax1.set_ylabel('测试准确率 (%)')
        ax1.set_title('不同选择比例下的准确率')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. 性能下降比较
        ax2 = axes[0, 1]
        for method in methods:
            method_data = ds_data[ds_data['method'] == method]
            if len(method_data) > 0:
                ratios = method_data['selection_ratio'] * 100
                drops = method_data['performance_drop']
                ax2.plot(ratios, drops, marker='s', label=method.upper(), linewidth=2)
        
        ax2.set_xlabel('Coreset大小 (%)')
        ax2.set_ylabel('性能下降 (%)')
        ax2.set_title('性能下降比较（越低越好）')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. 选择时间比较
        ax3 = axes[1, 0]
        for method in methods:
            method_data = ds_data[ds_data['method'] == method]
            if len(method_data) > 0:
                ratios = method_data['selection_ratio'] * 100
                times = method_data['selection_time']
                ax3.plot(ratios, times, marker='^', label=method.upper(), linewidth=2)
        
        ax3.set_xlabel('Coreset大小 (%)')
        ax3.set_ylabel('选择时间 (秒)')
        ax3.set_title('计算时间比较')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        ax3.set_yscale('log')
        
        # 4. 训练时间比较
        ax4 = axes[1, 1]
        for method in methods:
            method_data = ds_data[ds_data['method'] == method]
            if len(method_data) > 0:
                ratios = method_data['selection_ratio'] * 100
                times = method_data['train_time_coreset']
                ax4.plot(ratios, times, marker='d', label=method.upper(), linewidth=2)
        
        ax4.set_xlabel('Coreset大小 (%)')
        ax4.set_ylabel('训练时间 (秒)')
        ax4.set_title('训练时间比较')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("没有数据可绘制")

## 3. 持续学习实验分析

In [ ]:
# 将结果转换为DataFrame
if continual_results:
    df_continual = pd.DataFrame(continual_results)
    print("持续学习实验结果列:")
    print(df_continual.columns.tolist())
    print(f"\n结果数量: {len(df_continual)}")
    print(f"\n数据集: {df_continual['dataset'].unique().tolist()}")
    print(f"选择方法: {df_continual['selection_method'].unique().tolist()}")
else:
    print("没有持续学习实验结果")
    df_continual = pd.DataFrame()

In [ ]:
# 查看基本统计信息
if not df_continual.empty:
    print("\n基本统计信息:")
    print(df_continual.describe())
    
    # 按数据集和方法分组统计
    print("\n按数据集和方法分组:")
    summary = df_continual.groupby(['dataset', 'selection_method']).agg({
        'average_accuracy': ['mean', 'std'],
        'forgetting_measure': ['mean', 'std']
    })
    print(summary)

In [ ]:
# 绘制持续学习结果比较图
if not df_continual.empty:
    datasets = df_continual['dataset'].unique()
    
    for dataset in datasets:
        ds_data = df_continual[df_continual['dataset'] == dataset]
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f'{dataset} 数据集 - 持续学习性能', fontsize=16, fontweight='bold')
        
        methods = ds_data['selection_method'].unique()
        
        # 1. 平均准确率比较
        ax1 = axes[0, 0]
        for method in methods:
            method_data = ds_data[ds_data['selection_method'] == method]
            if len(method_data) > 0:
                x_pos = range(len(method_data))
                avg_accs = method_data['average_accuracy']
                ax1.plot(x_pos, avg_accs, marker='o', label=method.upper(), linewidth=2)
        
        ax1.set_xlabel('实验运行')
        ax1.set_ylabel('平均准确率 (%)')
        ax1.set_title('平均准确率比较')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. 遗忘度量比较
        ax2 = axes[0, 1]
        for method in methods:
            method_data = ds_data[ds_data['selection_method'] == method]
            if len(method_data) > 0:
                x_pos = range(len(method_data))
                forgettings = method_data['forgetting_measure']
                ax2.plot(x_pos, forgettings, marker='s', label=method.upper(), linewidth=2)
        
        ax2.set_xlabel('实验运行')
        ax2.set_ylabel('遗忘度量 (%)')
        ax2.set_title('遗忘度量比较（越低越好）')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 3. 准确率矩阵热图（选择第一个结果）
        ax3 = axes[1, 0]
        if len(ds_data) > 0:
            first_result = ds_data.iloc[0]
            acc_matrix = np.array(first_result['accuracy_matrix'])
            if acc_matrix.size > 0:
                num_tasks = acc_matrix.shape[0]
                im = ax3.imshow(acc_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=100)
                ax3.set_xticks(range(num_tasks))
                ax3.set_yticks(range(num_tasks))
                ax3.set_xlabel('任务ID')
                ax3.set_ylabel('学习任务')
                ax3.set_title(f'准确率矩阵 - {first_result["selection_method"].upper()}')
                
                # 添加数值标注
                for i in range(num_tasks):
                    for j in range(num_tasks):
                        if j <= i:
                            text = ax3.text(j, i, f'{acc_matrix[i, j]:.1f}',
                                          ha="center", va="center", color="black", fontsize=8)
                
                plt.colorbar(im, ax=ax3, label='准确率 (%)')
        
        # 4. 任务间准确率演变
        ax4 = axes[1, 1]
        if len(ds_data) > 0:
            first_result = ds_data.iloc[0]
            acc_matrix = np.array(first_result['accuracy_matrix'])
            if acc_matrix.size > 0:
                num_tasks = acc_matrix.shape[0]
                for task_id in range(num_tasks):
                    task_accuracies = []
                    for learned_task in range(task_id, num_tasks):
                        task_accuracies.append(acc_matrix[learned_task, task_id])
                    
                    x_pos = range(task_id, num_tasks)
                    ax4.plot(x_pos, task_accuracies, marker='o',
                            label=f'Task {task_id}', linewidth=2, alpha=0.7)
                
                ax4.set_xlabel('已学习的任务数')
                ax4.set_ylabel('准确率 (%)')
                ax4.set_title('各任务准确率演变')
                ax4.legend(ncol=2)
                ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("没有数据可绘制")

## 4. 创建比较表格

In [ ]:
# 数据摘要实验表格
if not df_data_summ.empty:
    key_cols = ['dataset', 'method', 'selection_ratio', 'coreset_size',
               'test_acc_full', 'test_acc_coreset', 'performance_drop',
               'selection_time', 'train_time_coreset']
    
    table_cols = [col for col in key_cols if col in df_data_summ.columns]
    table_df = df_data_summ[table_cols].copy()
    
    # 格式化
    if 'selection_ratio' in table_df.columns:
        table_df['selection_ratio'] = table_df['selection_ratio'] * 100
    
    # 重命名
    table_df.rename(columns={
        'dataset': 'Dataset',
        'method': 'Method',
        'selection_ratio': 'Ratio (%)',
        'coreset_size': 'Size',
        'test_acc_full': 'Full Acc (%)',
        'test_acc_coreset': 'Coreset Acc (%)',
        'performance_drop': 'Drop (%)',
        'selection_time': 'Sel Time (s)',
        'train_time_coreset': 'Train Time (s)'
    }, inplace=True)
    
    table_df = table_df.sort_values(['Dataset', 'Method', 'Ratio (%)'])
    
    print("数据摘要实验结果表:")
    print(table_df.to_string(index=False))
    
    # 保存为CSV
    output_dir = project_root / 'tables'
    output_dir.mkdir(exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_file = output_dir / f'data_summarization_{timestamp}.csv'
    table_df.to_csv(csv_file, index=False, float_format='%.2f')
    print(f"\n表格已保存: {csv_file}")
else:
    print("没有数据摘要实验结果")

In [ ]:
# 持续学习实验表格
if not df_continual.empty:
    key_cols = ['dataset', 'selection_method', 'memory_size',
               'num_tasks', 'average_accuracy', 'forgetting_measure']
    
    table_cols = [col for col in key_cols if col in df_continual.columns]
    table_df = df_continual[table_cols].copy()
    
    # 重命名
    table_df.rename(columns={
        'dataset': 'Dataset',
        'selection_method': 'Method',
        'memory_size': 'Memory',
        'num_tasks': 'Tasks',
        'average_accuracy': 'Avg Acc (%)',
        'forgetting_measure': 'Forgetting (%)'
    }, inplace=True)
    
    table_df = table_df.sort_values(['Dataset', 'Method'])
    
    print("持续学习实验结果表:")
    print(table_df.to_string(index=False))
    
    # 保存为CSV
    output_dir = project_root / 'tables'
    output_dir.mkdir(exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_file = output_dir / f'continual_learning_{timestamp}.csv'
    table_df.to_csv(csv_file, index=False, float_format='%.2f')
    print(f"\n表格已保存: {csv_file}")
else:
    print("没有持续学习实验结果")

## 5. 深入分析

这里可以进行更深入的分析，例如：
- 统计显著性检验
- 错误分析
- 特定案例研究
- 趋势分析

In [ ]:
# 示例：统计检验 - 比较不同方法的性能
from scipy import stats

if not df_data_summ.empty:
    print("\n统计显著性检验")
    print("="*60)
    
    methods = df_data_summ['method'].unique()
    
    # 选择特定选择比例的结果
    target_ratio = 0.1
    subset = df_data_summ[df_data_summ['selection_ratio'] == target_ratio]
    
    if len(subset) > 0 and len(methods) >= 2:
        print(f"\n比较选择比例 {target_ratio*100}% 下的性能:")
        
        # 成对t检验
        for i, method1 in enumerate(methods):
            for method2 in methods[i+1:]:
                data1 = subset[subset['method'] == method1]['test_acc_coreset']
                data2 = subset[subset['method'] == method2]['test_acc_coreset']
                
                if len(data1) > 1 and len(data2) > 1:
                    t_stat, p_value = stats.ttest_ind(data1, data2)
                    print(f"\n{method1.upper()} vs {method2.upper()}:")
                    print(f"  t统计量: {t_stat:.4f}")
                    print(f"  p值: {p_value:.4f}")
                    print(f"  显著性: {'是' if p_value < 0.05 else '否'} (α=0.05)")
else:
    print("没有足够的数据进行统计检验")

In [ ]:
# 示例：相关性分析
if not df_data_summ.empty:
    print("\n相关性分析")
    print("="*60)
    
    # 计算相关性矩阵
    numeric_cols = ['coreset_size', 'test_acc_coreset', 'performance_drop',
                   'selection_time', 'train_time_coreset']
    
    available_cols = [col for col in numeric_cols if col in df_data_summ.columns]
    
    if len(available_cols) > 1:
        corr_matrix = df_data_summ[available_cols].corr()
        print("\n相关性矩阵:")
        print(corr_matrix)
        
        # 绘制热图
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
                   fmt='.2f', linewidths=0.5)
        plt.title('特征相关性热图')
        plt.tight_layout()
        plt.show()
else:
    print("没有足够的数据进行相关性分析")

## 6. 导出报告

将分析结果导出为报告文件。

In [ ]:
# 创建分析报告
def create_report(data_summ_df, continual_df, output_path):
    """
    创建分析报告
    
    参数:
        data_summ_df: 数据摘要实验DataFrame
        continual_df: 持续学习实验DataFrame
        output_path: 输出文件路径
    """
    report_lines = []
    report_lines.append("# Coreset选择方法实验分析报告")
    report_lines.append(f"\n生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append("\n---\n")
    
    # 数据摘要实验
    if not data_summ_df.empty:
        report_lines.append("## 1. 数据摘要实验")
        report_lines.append(f"\n实验数量: {len(data_summ_df)}")
        report_lines.append(f"\n数据集: {', '.join(data_summ_df['dataset'].unique())}")
        report_lines.append(f"\n方法: {', '.join(data_summ_df['method'].unique())}")
        
        # 最佳结果
        best_idx = data_summ_df['test_acc_coreset'].idxmax()
        best_result = data_summ_df.loc[best_idx]
        report_lines.append(f"\n### 最佳结果")
        report_lines.append(f"- 数据集: {best_result['dataset']}")
        report_lines.append(f"- 方法: {best_result['method']}")
        report_lines.append(f"- 准确率: {best_result['test_acc_coreset']:.2f}%")
        report_lines.append(f"- 性能下降: {best_result['performance_drop']:.2f}%")
        report_lines.append("\n---\n")
    
    # 持续学习实验
    if not continual_df.empty:
        report_lines.append("## 2. 持续学习实验")
        report_lines.append(f"\n实验数量: {len(continual_df)}")
        report_lines.append(f"\n数据集: {', '.join(continual_df['dataset'].unique())}")
        report_lines.append(f"\n方法: {', '.join(continual_df['selection_method'].unique())}")
        
        # 最佳结果
        best_idx = continual_df['average_accuracy'].idxmax()
        best_result = continual_df.loc[best_idx]
        report_lines.append(f"\n### 最佳结果")
        report_lines.append(f"- 数据集: {best_result['dataset']}")
        report_lines.append(f"- 方法: {best_result['selection_method']}")
        report_lines.append(f"- 平均准确率: {best_result['average_accuracy']:.2f}%")
        report_lines.append(f"- 遗忘度量: {best_result['forgetting_measure']:.2f}%")
        report_lines.append("\n---\n")
    
    # 写入文件
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(report_lines))
    
    print(f"\n报告已保存: {output_path}")

# 创建报告
report_path = project_root / 'tables' / f'analysis_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.md'
create_report(df_data_summ, df_continual, report_path)

## 总结

本Notebook提供了完整的实验结果分析流程：

1. **数据加载**: 从JSON文件加载实验结果
2. **描述性统计**: 查看基本统计信息和分组统计
3. **可视化**: 创建多种比较图表
4. **表格生成**: 导出结果表格
5. **深入分析**: 统计检验和相关性分析
6. **报告生成**: 创建分析报告

可以根据需要修改和扩展分析内容。